# Initialization

In [1]:
import os
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [2]:
import sys

sys.path.insert(0, f'{project_dir}/code/Packages')

In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from glob import glob
import igraph as ig

---

In [4]:
import preprocessing
preprocessing.set_export_folder("final_edgelists")

Export directory set to C:\Users\ASUS\Downloads\SOURCE/data/3 - Network Generation/final_edgelists.


In [5]:
from importlib import reload

In [6]:
from wsi import graph_filtering as gf
from wsi import community_detection as cd
from wsi import network_conversion as nc

---

In [7]:
target_words_30 = ['pinto', 'sikat', 'dilim', 'sinag', 'kulay', 'sarap', 'tali', 'bayad', 'buto', 'suka', 'sumpa', 'patay', 'kupas', 'sipa', 'lason', 'salita', 'santo', 'lunod', 'alam', 'parusa', 'dahon', 'bato', 'dahan', 'ginhawa', 'susi', 'sulat', 'kulong', 'kalat', 'buwan', 'hamon',] 

In [9]:
path = f'{project_dir}/data/4 - Word Sense Induction'

words_df = pd.read_excel(f'{path}/words.xlsx')
with open(f'{path}/skip_this_words.txt') as f:
    exclude_txt = [word.strip() for word in f.readlines()]

In [10]:
target_words_borra = list(words_df[words_df['lemma'].apply(lambda word: len(str(word).split(" ")) == 1)]['lemma'])

In [11]:
target_words_all = list(set(target_words_30 + target_words_borra) - set(exclude_txt))

In [12]:
target_words_all = [word for word in target_words_all if type(word) == str]

In [13]:
target_words_all = [word.lower() for word in target_words_all]

---

In [14]:
graph = ig.read(f"{project_dir}/data/3 - Network Generation/aggregated_network.graphml", format="graphml")
graph.delete_vertices([vertex.index for vertex in graph.vs.select(_degree_eq = 0)])

In [15]:
print(f'Number of nodes: {len(graph.vs)}')

Number of nodes: 38221


In [16]:
print(f'Number of edges: {len(graph.es)}')

Number of edges: 388737


## WSI

In [17]:
index = 0
max_len = len(target_words_all[0:10])
for word in target_words_all[0:10]:
    print(f"{index + 1} over {max_len} ({int(((index + 1)/max_len) * 100)}%) {word}", end="")
    
    if graph.vs.select(word_eq=word):
        ego_network = gf.get_ego_network_from_word_list(graph, [word], True)

        # FILTER BASED ON FREQUENCY
        ego_index = ego_network.vs.select(word_eq=word)[0].index

        # get list of all weights of edges between ego and alter
        weights = [int(sum(ego_network.es.select(_source=neighbor_vertex.index, _target=ego_index)['weight'])) for neighbor_vertex in ego_network.vs if neighbor_vertex.index != ego_index]

        # get threshold based on percentile
        threshold = np.percentile(weights, 90)
        print(f" - {threshold}")

        # filter out those less than or equal to 15th percentile
        vertices_to_delete = [neighbor_vertex for neighbor_vertex in ego_network.vs if neighbor_vertex.index != ego_index and int(sum(ego_network.es.select(_source=neighbor_vertex.index, _target=ego_index)['weight'])) <= threshold]
        ego_network.delete_vertices(vertices_to_delete)
        ego_network.delete_vertices(ego_network.vs.select(word_eq=word))

        comms = cd.get_node_comm(cd.leiden_modularity_algorithm(ego_network, weighted=True))
        cd.save_communities(ego_network, graph, word, "leiden_mod", comms)

#         comms = cd.get_node_comm(cd.leiden_cpm_algorithm(ego_network, 0.5, weighted=True))
#         cd.save_communities(ego_network, graph, word, "leiden_cpm_algorithm", comms)

#         comms = cd.get_node_comm(cd.cw_algorithm(nc.convert_to_networkx(ego_network), 6))
#         cd.save_communities(ego_network, graph, word, "cw_algorithm", comms)

#         comms = cd.get_node_comm(cd.louvain_algorithm(ego_network, weighted=True))
#         cd.save_communities(ego_network, graph, word, "louvain", comms)

    else:
        print(f" - skipped.")
    index += 1

1 over 10 (10%) caesar - skipped.
2 over 10 (20%) siguro - 2.0
3 over 10 (30%) pahiga - skipped.
4 over 10 (40%) bengga - skipped.
5 over 10 (50%) ninkhursag - skipped.
6 over 10 (60%) granada - skipped.
7 over 10 (70%) harriman - skipped.
8 over 10 (80%) putin - 1.0
9 over 10 (90%) potyokin - skipped.
10 over 10 (100%) tugtog - 1.0
